# 🛰️ Route Resilience AI — Road Segmentation Training
## SegFormer-B2 for Coimbatore, Tamil Nadu

This notebook:
1. **Authenticates Google Earth Engine** and downloads Sentinel-2 imagery over Coimbatore
2. **Downloads road labels** from OpenStreetMap using OSMnx
3. **Rasterizes road vectors** into binary masks
4. **Trains SegFormer-B2** with Dice+BCE loss until **90% IoU accuracy**
5. **Saves the best model** to Google Drive

### Prerequisites
- Upload `reroute_colab.zip` to your Google Drive root folder
- Use **GPU runtime** (Runtime → Change Runtime → T4 GPU)

---
## Step 0: Check GPU & Install Dependencies

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU! Go to Runtime → Change Runtime Type → T4 GPU")

In [ ]:
!pip install -q earthengine-api osmnx geopandas rasterio transformers albumentations accelerate tqdm Pillow

---
## Step 1: Mount Google Drive & Unzip Model Code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive'
ZIP_PATH = os.path.join(DRIVE_ROOT, 'reroute_colab.zip')
WORK_DIR = '/content/reroute'
WEIGHTS_DIR = os.path.join(DRIVE_ROOT, 'reroute_weights')

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

# Unzip model code
if os.path.exists(ZIP_PATH):
    !unzip -o "{ZIP_PATH}" -d "{WORK_DIR}"
    print("✅ Model code extracted")
else:
    print(f"❌ Upload reroute_colab.zip to Drive root! Expected at: {ZIP_PATH}")

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)

# Verify imports
from models.segformer_road import SegFormerRoad, CombinedLoss
from augmentations.augment_config import get_train_transforms, get_val_transforms
print("✅ Model and augmentation code imported successfully")

---
## Step 2: Authenticate Google Earth Engine

In [ ]:
import ee

# This will open a browser popup to authenticate
ee.Authenticate()

# Initialize with your project (replace with your GEE project ID)
# You can find it at: https://console.cloud.google.com
try:
    ee.Initialize(project='ee-project')  # Try default
except:
    ee.Initialize()  # Fallback

print("✅ Google Earth Engine authenticated and initialized")

---
## Step 3: Download Sentinel-2 Imagery for Coimbatore

In [ ]:
import requests
import numpy as np
from PIL import Image
from io import BytesIO
import json

# ── Coimbatore Area of Interest ──────────────────────────
# City center: 11.0168°N, 76.9558°E
# We'll download a grid of tiles covering the city

COIMBATORE_CENTER = (11.0168, 76.9558)
TILE_SIZE_DEG = 0.02  # ~2.2km per tile at this latitude
GRID_SIZE = 8         # 8×8 grid = 64 tiles
IMAGE_SIZE = 512      # pixels per tile

DATA_DIR = '/content/dataset'
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
MASKS_DIR = os.path.join(DATA_DIR, 'masks')
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(MASKS_DIR, exist_ok=True)

print(f"📍 Coimbatore center: {COIMBATORE_CENTER}")
print(f"📦 Grid: {GRID_SIZE}×{GRID_SIZE} = {GRID_SIZE**2} tiles")
print(f"🖼️ Each tile: {IMAGE_SIZE}×{IMAGE_SIZE} pixels")

In [ ]:
from tqdm import tqdm

def get_sentinel2_tile(bbox, tile_id):
    """
    Download a cloud-masked Sentinel-2 RGB tile from GEE.
    bbox: [west, south, east, north]
    """
    region = ee.Geometry.Rectangle(bbox)
    
    # Cloud masking function
    def mask_clouds(image):
        qa = image.select('QA60')
        cloud_mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
        return image.updateMask(cloud_mask)
    
    # Get best cloud-free composite (last 6 months)
    collection = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(region)
        .filterDate('2024-10-01', '2025-06-01')
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_clouds)
    )
    
    composite = collection.median().select(['B4', 'B3', 'B2'])  # RGB
    
    # Get download URL
    url = composite.getThumbURL({
        'region': region,
        'dimensions': f'{IMAGE_SIZE}x{IMAGE_SIZE}',
        'format': 'png',
        'min': 0,
        'max': 3000,
    })
    
    response = requests.get(url, timeout=60)
    if response.status_code == 200:
        img = Image.open(BytesIO(response.content)).convert('RGB')
        img = img.resize((IMAGE_SIZE, IMAGE_SIZE))
        return img
    else:
        print(f"  ⚠️ Failed tile {tile_id}: HTTP {response.status_code}")
        return None


# Generate tile grid over Coimbatore
lat_center, lon_center = COIMBATORE_CENTER
half_span = TILE_SIZE_DEG * GRID_SIZE / 2

tiles_downloaded = 0
tile_metadata = []

print("🛰️ Downloading Sentinel-2 tiles from Google Earth Engine...")
print(f"   Area: {lon_center - half_span:.4f}°E to {lon_center + half_span:.4f}°E")
print(f"          {lat_center - half_span:.4f}°N to {lat_center + half_span:.4f}°N")
print()

for row in tqdm(range(GRID_SIZE), desc="Rows"):
    for col in range(GRID_SIZE):
        tile_id = f"coimbatore_{row:02d}_{col:02d}"
        
        # Calculate bbox for this tile
        west = lon_center - half_span + col * TILE_SIZE_DEG
        south = lat_center - half_span + row * TILE_SIZE_DEG
        east = west + TILE_SIZE_DEG
        north = south + TILE_SIZE_DEG
        bbox = [west, south, east, north]
        
        save_path = os.path.join(IMAGES_DIR, f"{tile_id}.png")
        
        # Skip if already downloaded
        if os.path.exists(save_path):
            tiles_downloaded += 1
            tile_metadata.append({'id': tile_id, 'bbox': bbox})
            continue
        
        try:
            img = get_sentinel2_tile(bbox, tile_id)
            if img is not None:
                img.save(save_path)
                tiles_downloaded += 1
                tile_metadata.append({'id': tile_id, 'bbox': bbox})
        except Exception as e:
            print(f"  ❌ Error tile {tile_id}: {e}")
        
        # Small delay to avoid GEE rate limiting
        import time
        time.sleep(0.5)

print(f"\n✅ Downloaded {tiles_downloaded}/{GRID_SIZE**2} Sentinel-2 tiles")

# Save metadata
with open(os.path.join(DATA_DIR, 'tile_metadata.json'), 'w') as f:
    json.dump(tile_metadata, f, indent=2)

---
## Step 4: Download Road Labels from OpenStreetMap

In [ ]:
import osmnx as ox
import geopandas as gpd
from shapely.geometry import box
import cv2

print("🗺️ Downloading road network from OpenStreetMap for Coimbatore...")

# Download full Coimbatore road network
lat_center, lon_center = COIMBATORE_CENTER
half_span = TILE_SIZE_DEG * GRID_SIZE / 2

# Get roads for the entire area
try:
    G = ox.graph_from_bbox(
        lat_center + half_span + 0.005,   # north
        lat_center - half_span - 0.005,   # south  
        lon_center + half_span + 0.005,   # east
        lon_center - half_span - 0.005,   # west
        network_type='drive'
    )
    edges = ox.graph_to_gdfs(G, nodes=False)
    print(f"✅ Downloaded {len(edges)} road segments from OSM")
except Exception as e:
    print(f"Trying alternative download method...")
    G = ox.graph_from_point(COIMBATORE_CENTER, dist=10000, network_type='drive')
    edges = ox.graph_to_gdfs(G, nodes=False)
    print(f"✅ Downloaded {len(edges)} road segments from OSM")

In [ ]:
from shapely.geometry import box as shapely_box
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from shapely.ops import unary_union

def create_road_mask(edges_gdf, bbox, image_size=512, buffer_m=3):
    """
    Create a binary road mask from OSM road vectors.
    Buffers road lines by ~3 meters to create visible road width.
    """
    west, south, east, north = bbox
    tile_box = shapely_box(west, south, east, north)
    
    # Clip roads to tile extent
    clipped = edges_gdf[edges_gdf.intersects(tile_box)].copy()
    
    if len(clipped) == 0:
        return np.zeros((image_size, image_size), dtype=np.uint8)
    
    # Buffer roads (convert degrees to approximate meters: 1° ≈ 111km at equator)
    buffer_deg = buffer_m / 111000.0
    
    # Apply different buffer widths based on road type
    geometries = []
    for _, row in clipped.iterrows():
        highway = row.get('highway', 'residential')
        if isinstance(highway, list):
            highway = highway[0]
        
        # Wider roads get bigger buffers
        if highway in ('motorway', 'trunk', 'primary'):
            buf = buffer_deg * 3
        elif highway in ('secondary', 'tertiary'):
            buf = buffer_deg * 2
        else:
            buf = buffer_deg
        
        buffered = row.geometry.buffer(buf)
        clipped_geom = buffered.intersection(tile_box)
        if not clipped_geom.is_empty:
            geometries.append(clipped_geom)
    
    if len(geometries) == 0:
        return np.zeros((image_size, image_size), dtype=np.uint8)
    
    # Rasterize
    transform = from_bounds(west, south, east, north, image_size, image_size)
    mask = rasterize(
        [(geom, 1) for geom in geometries],
        out_shape=(image_size, image_size),
        transform=transform,
        fill=0,
        dtype=np.uint8
    )
    
    return mask


print("🏗️ Generating road masks for each satellite tile...")

masks_created = 0
for meta in tqdm(tile_metadata, desc="Creating masks"):
    tile_id = meta['id']
    bbox = meta['bbox']
    
    mask_path = os.path.join(MASKS_DIR, f"{tile_id}.png")
    if os.path.exists(mask_path):
        masks_created += 1
        continue
    
    mask = create_road_mask(edges, bbox, IMAGE_SIZE)
    
    # Only save if mask has some road pixels (skip empty tiles)
    road_pct = mask.sum() / mask.size * 100
    if road_pct > 0.5:  # At least 0.5% road coverage
        Image.fromarray(mask * 255).save(mask_path)
        masks_created += 1
    else:
        # Remove the satellite image too if no roads
        img_path = os.path.join(IMAGES_DIR, f"{tile_id}.png")
        if os.path.exists(img_path):
            os.remove(img_path)

print(f"\n✅ Created {masks_created} road masks (tiles with road coverage > 0.5%)")

In [ ]:
# Visualize some samples
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Coimbatore — Satellite Image / Road Mask Pairs', fontsize=14)

mask_files = sorted([f for f in os.listdir(MASKS_DIR) if f.endswith('.png')])[:6]

for i, fname in enumerate(mask_files):
    img = Image.open(os.path.join(IMAGES_DIR, fname))
    mask = Image.open(os.path.join(MASKS_DIR, fname))
    
    axes[i // 2, (i % 2) * 2].imshow(img)
    axes[i // 2, (i % 2) * 2].set_title(f'Satellite: {fname}')
    axes[i // 2, (i % 2) * 2].axis('off')
    
    axes[i // 2, (i % 2) * 2 + 1].imshow(mask, cmap='hot')
    axes[i // 2, (i % 2) * 2 + 1].set_title(f'Road Mask')
    axes[i // 2, (i % 2) * 2 + 1].axis('off')

plt.tight_layout()
plt.show()

---
## Step 5: Create PyTorch Dataset & DataLoaders

In [ ]:
from torch.utils.data import Dataset, DataLoader
import glob

class RoadSegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform
        
        # Only include images that have corresponding masks
        mask_files = set(os.listdir(masks_dir))
        self.filenames = sorted([
            f for f in os.listdir(images_dir)
            if f.endswith('.png') and f in mask_files
        ])
        print(f"  Dataset: {len(self.filenames)} image-mask pairs")
    
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        fname = self.filenames[idx]
        
        image = np.array(Image.open(os.path.join(self.images_dir, fname)).convert('RGB'))
        mask = np.array(Image.open(os.path.join(self.masks_dir, fname)).convert('L'))
        
        # Normalize mask to 0-1
        mask = (mask > 127).astype(np.float32)
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']  # [3, H, W] tensor
            mask = augmented['mask']    # [H, W] tensor
        
        # Add channel dim to mask: [H, W] → [1, H, W]
        mask = mask.unsqueeze(0) if isinstance(mask, torch.Tensor) else torch.tensor(mask).unsqueeze(0)
        
        return image, mask.float()


# Split dataset: 80% train, 20% val
all_masks = sorted([f for f in os.listdir(MASKS_DIR) if f.endswith('.png')])
n_total = len(all_masks)
n_train = int(0.8 * n_total)

# Shuffle and split
np.random.seed(42)
indices = np.random.permutation(n_total)
train_files = [all_masks[i] for i in indices[:n_train]]
val_files = [all_masks[i] for i in indices[n_train:]]

# Create separate dirs for train/val
for split, files in [('train', train_files), ('val', val_files)]:
    split_img_dir = os.path.join(DATA_DIR, split, 'images')
    split_mask_dir = os.path.join(DATA_DIR, split, 'masks')
    os.makedirs(split_img_dir, exist_ok=True)
    os.makedirs(split_mask_dir, exist_ok=True)
    
    for f in files:
        src_img = os.path.join(IMAGES_DIR, f)
        src_mask = os.path.join(MASKS_DIR, f)
        if os.path.exists(src_img) and os.path.exists(src_mask):
            import shutil
            shutil.copy2(src_img, os.path.join(split_img_dir, f))
            shutil.copy2(src_mask, os.path.join(split_mask_dir, f))

print(f"\n📊 Dataset split:")
print(f"   Train: {len(train_files)} tiles")
print(f"   Val:   {len(val_files)} tiles")

In [ ]:
# Create datasets with augmentation
CROP_SIZE = 512
BATCH_SIZE = 4

train_dataset = RoadSegmentationDataset(
    os.path.join(DATA_DIR, 'train', 'images'),
    os.path.join(DATA_DIR, 'train', 'masks'),
    transform=get_train_transforms(CROP_SIZE)
)

val_dataset = RoadSegmentationDataset(
    os.path.join(DATA_DIR, 'val', 'images'),
    os.path.join(DATA_DIR, 'val', 'masks'),
    transform=get_val_transforms(CROP_SIZE)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n✅ DataLoaders ready")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches:   {len(val_loader)}")

---
## Step 6: Initialize Model & Training Setup

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Training on: {device}")

# Initialize model
model = SegFormerRoad(pretrained=True)
model = model.to(device)
print(f"📐 Trainable parameters: {model.count_parameters():,}")

# Loss, optimizer, scheduler
criterion = CombinedLoss(dice_weight=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=6e-5, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

# Mixed precision training for speed
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

print("✅ Model, optimizer, and scheduler initialized")

---
## Step 7: Training Loop — Target: 90% IoU

In [ ]:
def calculate_iou(logits, targets, threshold=0.5):
    """Calculate Intersection over Union (IoU) for binary segmentation."""
    preds = (torch.sigmoid(logits) > threshold).float()
    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection
    if union == 0:
        return 1.0  # Both empty
    return (intersection / union).item()


def calculate_accuracy(logits, targets, threshold=0.5):
    """Calculate pixel-wise accuracy."""
    preds = (torch.sigmoid(logits) > threshold).float()
    correct = (preds == targets).float().sum()
    total = targets.numel()
    return (correct / total).item()


def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = 0
    total_iou = 0
    total_acc = 0
    n_batches = 0
    
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        total_loss += loss.item()
        total_iou += calculate_iou(logits.detach(), masks)
        total_acc += calculate_accuracy(logits.detach(), masks)
        n_batches += 1
    
    return total_loss / n_batches, total_iou / n_batches, total_acc / n_batches


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_iou = 0
    total_acc = 0
    n_batches = 0
    
    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)
        
        logits = model(images)
        loss = criterion(logits, masks)
        
        total_loss += loss.item()
        total_iou += calculate_iou(logits, masks)
        total_acc += calculate_accuracy(logits, masks)
        n_batches += 1
    
    return total_loss / n_batches, total_iou / n_batches, total_acc / n_batches

In [ ]:
# ═══════════════════════════════════════════════════════════
#  TRAINING LOOP — Trains until 90% IoU or max 100 epochs
# ═══════════════════════════════════════════════════════════

TARGET_IOU = 0.90
MAX_EPOCHS = 100
PATIENCE = 15  # Early stopping patience

best_iou = 0.0
best_epoch = 0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_iou': [], 'val_iou': [], 'train_acc': [], 'val_acc': []}

print("="*70)
print(f"🚀 TRAINING SegFormer-B2 for Coimbatore Road Segmentation")
print(f"   Target IoU: {TARGET_IOU*100:.0f}% | Max Epochs: {MAX_EPOCHS} | Patience: {PATIENCE}")
print("="*70)

for epoch in range(1, MAX_EPOCHS + 1):
    # Train
    train_loss, train_iou, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device
    )
    
    # Validate
    val_loss, val_iou, val_acc = validate(model, val_loader, criterion, device)
    
    # Step scheduler
    scheduler.step(epoch)
    
    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_iou'].append(train_iou)
    history['val_iou'].append(val_iou)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    lr = optimizer.param_groups[0]['lr']
    
    # Print progress
    print(
        f"Epoch {epoch:3d}/{MAX_EPOCHS} │ "
        f"Train Loss: {train_loss:.4f} IoU: {train_iou:.4f} Acc: {train_acc:.4f} │ "
        f"Val Loss: {val_loss:.4f} IoU: {val_iou:.4f} Acc: {val_acc:.4f} │ "
        f"LR: {lr:.2e}"
    )
    
    # Save best model
    if val_iou > best_iou:
        best_iou = val_iou
        best_epoch = epoch
        patience_counter = 0
        
        # Save to Drive
        save_path = os.path.join(WEIGHTS_DIR, 'best_model.pth')
        torch.save(model.state_dict(), save_path)
        print(f"        💾 New best model saved! IoU: {best_iou:.4f}")
        
        # Also save locally
        torch.save(model.state_dict(), '/content/best_model.pth')
    else:
        patience_counter += 1
    
    # Check if we hit target
    if val_iou >= TARGET_IOU:
        print(f"\n🎯 TARGET REACHED! Val IoU: {val_iou:.4f} ≥ {TARGET_IOU:.4f}")
        print(f"   Model saved to: {WEIGHTS_DIR}/best_model.pth")
        break
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\n⏹️ Early stopping at epoch {epoch}. Best IoU: {best_iou:.4f} at epoch {best_epoch}")
        break

print(f"\n{'='*70}")
print(f"🏆 Training Complete!")
print(f"   Best Val IoU:  {best_iou:.4f} (Epoch {best_epoch})")
print(f"   Model saved:   {WEIGHTS_DIR}/best_model.pth")
print(f"{'='*70}")

---
## Step 8: Training Curves & Evaluation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', color='#3b82f6')
axes[0].plot(history['val_loss'], label='Val', color='#ef4444')
axes[0].set_title('Loss (Dice + BCE)', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU
axes[1].plot(history['train_iou'], label='Train', color='#3b82f6')
axes[1].plot(history['val_iou'], label='Val', color='#ef4444')
axes[1].axhline(y=TARGET_IOU, color='#10b981', linestyle='--', label=f'Target ({TARGET_IOU*100:.0f}%)')
axes[1].set_title('IoU (Intersection over Union)', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Accuracy
axes[2].plot(history['train_acc'], label='Train', color='#3b82f6')
axes[2].plot(history['val_acc'], label='Val', color='#ef4444')
axes[2].set_title('Pixel Accuracy', fontsize=13)
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Route Resilience AI — Coimbatore Road Segmentation Training', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(WEIGHTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("📊 Training curves saved to Drive")

In [ ]:
# Visualize predictions on validation set
model.eval()

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
columns = ['Satellite Image', 'Ground Truth Mask', 'Model Prediction']
for ax, col in zip(axes[0], columns):
    ax.set_title(col, fontsize=13, fontweight='bold')

val_iter = iter(val_loader)
for row in range(4):
    try:
        images, masks = next(val_iter)
    except StopIteration:
        break
    
    images_gpu = images.to(device)
    with torch.no_grad():
        logits = model(images_gpu)
        preds = (torch.sigmoid(logits) > 0.5).float().cpu()
    
    # Denormalize image for display
    img = images[0].permute(1, 2, 0).numpy()
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    
    axes[row, 0].imshow(img)
    axes[row, 0].axis('off')
    
    axes[row, 1].imshow(masks[0, 0].numpy(), cmap='hot')
    axes[row, 1].axis('off')
    
    axes[row, 2].imshow(preds[0, 0].numpy(), cmap='hot')
    axes[row, 2].axis('off')

plt.suptitle('Coimbatore Road Predictions — SegFormer-B2', fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(WEIGHTS_DIR, 'predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Step 9: Export Final Model

The trained model is saved at:
- **Google Drive**: `/My Drive/reroute_weights/best_model.pth`
- **Training curves**: `/My Drive/reroute_weights/training_curves.png`

Download `best_model.pth` and place it in your local project at:
```
e:\Reroute\ai\weights\best_model.pth
```

Then start the backend server:
```bash
cd e:\Reroute
.venv\Scripts\python backend\main.py
```

In [ ]:
# Final summary
print("\n" + "="*60)
print("  🎉 TRAINING COMPLETE — Route Resilience AI")
print("="*60)
print(f"  City:           Coimbatore, Tamil Nadu")
print(f"  Model:          SegFormer-B2")
print(f"  Parameters:     {model.count_parameters():,}")
print(f"  Best Val IoU:   {best_iou:.4f} ({best_iou*100:.1f}%)")
print(f"  Best Epoch:     {best_epoch}")
print(f"  Weights saved:  {WEIGHTS_DIR}/best_model.pth")
print("="*60)
print("\n📋 Next Steps:")
print("  1. Download best_model.pth from Google Drive")
print("  2. Place in: e:\\Reroute\\ai\\weights\\best_model.pth")
print("  3. Start backend: python backend/main.py")
print("  4. Start frontend: cd frontend && npm run dev")
print("  5. Open browser: http://localhost:5173")